In [43]:
import pandas as pd
import numpy as np

df = pd.read_csv('experiment_results.csv')

df['days'] = df['Elapsed'].str.split('-').str[0].where(df['Elapsed'].str.contains('-'), 0).astype(int)
df['hours'] = np.where(df['Elapsed'].str.contains('-'), df['Elapsed'].str.split('-').str[1].str.split(':').str[0], df['Elapsed'].str.split(':').str[0]).astype(int)
df['minutes'] = np.where(df['Elapsed'].str.contains('-'), df['Elapsed'].str.split('-').str[1].str.split(':').str[1], df['Elapsed'].str.split(':').str[1]).astype(int)
df['seconds'] = np.where(df['Elapsed'].str.contains('-'), df['Elapsed'].str.split('-').str[1].str.split(':').str[2], df['Elapsed'].str.split(':').str[2]).astype(int)

df['TotalFiles'] = df['NumFiles'].sum()

df['TotalHours'] = df['hours'] + df['minutes'] / 60 + (df['seconds']/60)/60 + df['days']*24

df['TotalWalltimeHours'] = df['TotalHours'].sum()

df['TotalWalltimeDays'] = df['TotalWalltimeHours'] / 24

df['MaxMemoryUsedNoUnit'] = df['MaxMemoryUsed'].str.replace('GB', '').astype(int)

df['TotalMemoryUsedGB'] = df['MaxMemoryUsedNoUnit'].sum()

df['TotalMemoryUsedTB'] = df['TotalMemoryUsedGB'].astype(int) / 1024
df['MaxMemoryUsedGB'] = df['MaxMemoryUsedNoUnit'].max()
df['AvgMemoryUsedGB'] = df['MaxMemoryUsedNoUnit'].mean()

df['MaxRSSGB'] = df['MaxRSS'].apply(lambda x: 
    (float(x[:-1]) / 1024 / 1024) if x.endswith('K') else 
    (float(x[:-1]) / 1024) if x.endswith('M') else
    float(x[:-1])  # assume G
)
df['AvgRSSGB'] = df['MaxRSSGB'].mean()
df['MaxMaxRSSGB'] = df['MaxRSSGB'].max()

df['TotalRSSGB'] = df['MaxRSSGB'].sum()

print(df[['TotalWalltimeDays', 'MaxMemoryUsedGB', 'AvgRSSGB', 'MaxMaxRSSGB']].head(1))

# sacct -j <job_id> --format=JobID,MaxRSS

   TotalWalltimeDays  MaxMemoryUsedGB    AvgRSSGB  MaxMaxRSSGB
0          14.140775              470  341.306638   437.810654


In [ ]:
df_files = pd.read_csv('parser_log_files.csv')

df_files.columns

df_files['TotalPeakMemoryGB'] = df_files['peak_memory_mb'].sum() / 1024

df_files['AvgPeakMemoryGBPerFile'] = df_files['peak_memory_mb'].mean() / 1024

df_files['MaxPeakMemoryGBPerFile'] = df_files['peak_memory_mb'].max() / 1024

df_files['TotalEntities'] = df_files['num_entities'].sum()

df_files['TotalProcessedRevisions'] = df_files['processed_revisions'].sum()

df_files['AvgProcessTimePerFileMinutes'] = (df_files['total_process_time_sec'].mean() / 60)

print(df_files[['AvgPeakMemoryGBPerFile', 'MaxPeakMemoryGBPerFile', 'TotalEntities', 'TotalProcessedRevisions', 'AvgProcessTimePerFileMinutes']].head(1))


   AvgPeakMemoryGBPerFile  MaxPeakMemoryGBPerFile  TotalEntities  \
0               36.245297              208.782238      120794825   

   TotalProcessedRevisions  AvgProcessTimePerFileMinutes  
0               2312515201                     46.968063  


In [ ]:
# Query to obtain metrics from processing

query = """SELECT 
            SUM(num_revisions) AS total_revisions,
            COUNT(*) AS total_entities,
            SUM(total_xml_parse_time_sec + total_process_time_sec) AS total_processing_time_sec,
            SUM(total_revision_diff_time_sec) AS total_diff_time_sec,
            SUM(num_revisions_timed) AS total_revisions_timed
        FROM (
            SELECT * FROM entity_stats_less
            UNION ALL
            SELECT * FROM entity_stats_sa
            UNION ALL
            SELECT * FROM entity_stats_ao
            UNION ALL
            SELECT * FROM entity_stats
        ) all_stats;"""

# total_revisions = 1384523689 <-
# total_entities = 120794825
# total_processing_time_sec = 14326748.256660938
# total_diff_time_sec = 5682724.07952261
# total_revisions_timed = 2312265238 <-

# The difference between total_revisions and total_revisions_timed is that the first one is the actual revisions we save, 
# and the latter is the number of revisions we don't save but still have to process (this is because we don't save revisions where there was an alias or sitelink change, or revisions where
# there was a change to a label/description in a language that is not english)

# avg revision diff time in sec: 5682724.07952261 / 2312265238 = 0.00245764369 seconds = 2.45764369 miliseconds
# throughput of entities: 120794825 / 14326748.256660938 = 8.43141952633 entities per second
# average throughput of revisions: 2312265238 / 14326748.256660938 = 161.394979278 revisions per second